# 01 — Batch preprocess

Runs the preprocessing pipeline (`raw → processed`) on all participants × all valid conditions.

## Pipeline

1. Load `.mat` and extract marker trajectories
2. Apply manual `crop_start_s` / `crop_end_s` from `participants.py`
3. `preprocess.run()`:
   - Dos01–04 barycentre (XYZ)
   - Despike (MAD, k=8) — linear interpolation of spike samples
   - EMD with adaptive IMF selection (walking band [0.5–2.5 Hz], fallback heuristic)
   - Butterworth low-pass (5 Hz, order 4, zero-phase)
   - Decimation 400 → 100 Hz (anti-aliased)
4. Save `.npz` via `writer.save_processed()`

## Behaviour

- `FORCE=True` → reprocess everything (overwrites existing `.npz`)
- `FORCE=False` → skip trials with an existing `.npz`
- Trials with `excluded=True` in `participants.py` are skipped automatically
- Per-trial errors are logged but do not stop the loop (summary at the end)

## Inputs / Outputs

- **Input**  : `data/raw/{participant}/*.mat`
- **Output** : `data/processed/{participant}/{participant}_{NN}_{condition}.npz`

In [1]:
from resilience import paths, participants, config
from resilience.io import loader, writer
from resilience.processing import preprocess

from collections import defaultdict
import numpy as np
import time

## Configuration

In [2]:
FORCE = False   # True = overwrite existing .npz / False = skip them
TRIALS = participants.list_trials_to_process()
# TRIALS = [("005DeJe", "silence")]

print(f"Trials to process: {len(TRIALS)}")
print(f"FORCE = {FORCE}\n")

by_p = defaultdict(list)
for p, c in TRIALS:
    by_p[p].append(c)
for p in sorted(by_p):
    print(f"  {p:10s} → {len(by_p[p])} trial(s)  ({', '.join(by_p[p])})")

Trials to process: 46
FORCE = False

  001CrMa    → 3 trial(s)  (beatmove_adaptatif, silence, tempo_random)
  002CrPa    → 3 trial(s)  (beatmove_adaptatif, silence, tempo_random)
  003BrLu    → 3 trial(s)  (beatmove_adaptatif, silence, tempo_random)
  004CaGe    → 3 trial(s)  (beatmove_adaptatif, silence, tempo_random)
  005DeJe    → 3 trial(s)  (beatmove_adaptatif, silence, tempo_random)
  006MoCa    → 3 trial(s)  (beatmove_adaptatif, silence, tempo_random)
  007MoMa    → 1 trial(s)  (tempo_random)
  008RiMo    → 3 trial(s)  (beatmove_adaptatif, silence, tempo_random)
  009DeFr    → 3 trial(s)  (beatmove_adaptatif, silence, tempo_random)
  010DeYv    → 3 trial(s)  (beatmove_adaptatif, silence, tempo_random)
  011RiJo    → 3 trial(s)  (beatmove_adaptatif, silence, tempo_random)
  012WaCh    → 3 trial(s)  (beatmove_adaptatif, silence, tempo_random)
  013WaJe    → 3 trial(s)  (beatmove_adaptatif, silence, tempo_random)
  014DeAn    → 3 trial(s)  (beatmove_adaptatif, silence, tempo_random

## Batch loop

In [3]:
results = {'ok': [], 'skipped': [], 'failed': []}
t_start = time.time()
current_p = None

for participant, condition in TRIALS:
    if participant != current_p:
        print(f"\n{'═'*60}\n  {participant}\n{'═'*60}")
        current_p = participant

    key = f"{participant} / {condition}"
    try:
        # 1. Skip if already processed
        if not FORCE:
            try:
                npz = paths.processed_file(participant, condition)
                print(f"  ⏭️  {condition:22s} → skip (exists: {npz.name})")
                results['skipped'].append(key)
                continue
            except FileNotFoundError:
                pass

        # 2. Locate raw .mat
        raw_path = paths.raw_file(participant, condition)
        print(f"  ▶️   {condition:22s} → {raw_path.name}")

        # 3. Load + extract trajectories
        data, fmt = loader.load_mat(raw_path)
        trajectories = loader.extract_marker_trajectories(data, fmt)
        if trajectories is None:
            raise RuntimeError("extract_marker_trajectories returned None")

        # 4. Manual crop (crop_start_s + crop_end_s from participants.py)
        trial = participants.get_trial(participant, condition)
        n_before = next(iter(trajectories.values())).shape[0]
        i_start = 0
        i_end   = n_before

        if trial.crop_start_s is not None:
            i_start = int(trial.crop_start_s * config.FS_RAW)
            print(f"      ✂️  crop_start_s = {trial.crop_start_s}s")
        if trial.crop_end_s is not None:
            i_end = int(trial.crop_end_s * config.FS_RAW)
            print(f"      ✂️  crop_end_s   = {trial.crop_end_s}s")

        if i_start > 0 or i_end < n_before:
            trajectories = {m: xyz[i_start:i_end] for m, xyz in trajectories.items()}
            n_after = i_end - i_start
            print(f"      → {n_before} frames → {n_after} frames ({n_after/config.FS_RAW:.1f}s)")

        # 5. Preprocess (despike + EMD + Butterworth + decimation)
        out = preprocess.run(trajectories, axis='Z', verbose=False)

        # 6. Save .npz
        npz_path = writer.save_processed(participant, condition, out,
                                          trajectories_raw=trajectories)
        print(f"      ✅ {npz_path.name}")
        results['ok'].append(key)

    except FileNotFoundError as e:
        print(f"      ⚠️  raw .mat not found")
        results['failed'].append((key, 'FileNotFoundError', str(e)))
    except Exception as e:
        print(f"      ❌ {type(e).__name__}: {e}")
        results['failed'].append((key, type(e).__name__, str(e)))

elapsed = time.time() - t_start
print(f"\n{'═'*60}")
print(f"  Elapsed: {elapsed/60:.1f} min  ({elapsed:.0f} s)")
print(f"{'═'*60}")


════════════════════════════════════════════════════════════
  001CrMa
════════════════════════════════════════════════════════════
  ⏭️  beatmove_adaptatif     → skip (exists: 001CrMa_03_beatmove_adaptatif.npz)
  ⏭️  silence                → skip (exists: 001CrMa_01_silence.npz)
  ⏭️  tempo_random           → skip (exists: 001CrMa_02_tempo_random.npz)

════════════════════════════════════════════════════════════
  002CrPa
════════════════════════════════════════════════════════════
  ⏭️  beatmove_adaptatif     → skip (exists: 002CrPa_02_beatmove_adaptatif.npz)
  ⏭️  silence                → skip (exists: 002CrPa_01_silence.npz)
  ⏭️  tempo_random           → skip (exists: 002CrPa_03_tempo_random.npz)

════════════════════════════════════════════════════════════
  003BrLu
════════════════════════════════════════════════════════════
  ⏭️  beatmove_adaptatif     → skip (exists: 003BrLu_01_beatmove_adaptatif.npz)
  ⏭️  silence                → skip (exists: 003BrLu_03_silence.npz)
  ⏭️  

## Summary

In [4]:
print(f"  ✅ OK       : {len(results['ok'])}")
print(f"  ⏭️  Skipped  : {len(results['skipped'])}")
print(f"  ❌ Failed   : {len(results['failed'])}")

if results['failed']:
    print(f"\n  Failures:")
    for key, err_type, err_msg in results['failed']:
        print(f"    [{err_type}] {key}")
        print(f"        → {err_msg[:120]}")

excluded = [(p.code, cond) for p in participants.PARTICIPANTS.values()
            for cond, t in p.trials.items() if t.excluded]
if excluded:
    print(f"\n  🚫 Excluded trials (excluded=True): {len(excluded)}")
    for p, c in excluded:
        trial = participants.get_trial(p, c)
        print(f"    {p} / {c}  —  {trial.note or 'no note'}")

  ✅ OK       : 6
  ⏭️  Skipped  : 40
  ❌ Failed   : 0

  🚫 Excluded trials (excluded=True): 2
    007MoMa / silence  —  EXCLUDE: broken back markers, cluster fill FOPT=28mm (non-rigid), amplitude 180mm, f_dom 0.75Hz. V2 confirms exclusion.
    007MoMa / beatmove_adaptatif  —  EXCLUDE: amplitude 213mm, f_dom 0.75Hz (not walking). V2 confirms exclusion.


## Quick check on one produced .npz

In [5]:
if results['ok']:
    p, c = results['ok'][0].split(' / ')
    payload = writer.load_processed(p, c)
    print(f"File: {p} / {c}\n")
    print("Keys:")
    for k in payload:
        arr = payload[k]
        if isinstance(arr, np.ndarray):
            print(f"    {k:22s} shape={str(arr.shape):20s} dtype={arr.dtype}")
        else:
            print(f"    {k:22s} = {arr}")

File: 015LaOd / beatmove_adaptatif

Keys:
    sacrum_xyz_raw         shape=(168001, 3)          dtype=float64
    sacrum_axis_raw        shape=(168001,)            dtype=float64
    sacrum_despiked        shape=(168001,)            dtype=float64
    sacrum_emd             shape=(168001,)            dtype=float64
    sacrum_filt            shape=(168001,)            dtype=float64
    signal_final           shape=(42001,)             dtype=float64
    time_final             shape=(42001,)             dtype=float64
    fs_final               shape=()                   dtype=int64
    axis                   shape=()                   dtype=<U1
    emd_applied            shape=()                   dtype=bool
    despike_applied        shape=()                   dtype=bool
    cluster_fill_applied   shape=()                   dtype=bool
    makima_applied         shape=()                   dtype=bool
    despike_stats          shape=()                   dtype=object
    cluster_fill_stats   